# GeoStat_py · Workbench analítico temporal (Colab)

Este notebook asume que ya corriste `colab/00_bootstrap.ipynb` y que el entorno tiene acceso al repo.

Flujo único:
1. Subir CSV local (uploader de Colab).
2. Cargar dataset y ver autodetección.
3. Configurar X/Y/Z/target.
4. Ejecutar EDA.
5. Ejecutar variografía básica.


In [ ]:
# 0) Preparación mínima de entorno y servicio (con autorecuperación de sys.path)
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_URL = "https://github.com/joelmanrique91-lgtm/GeoStat_py.git"
BRANCH = "main"
BASE_DIR = "/content"
REPO_DIR_NAME = "GeoStat_py"
REPO_DIR = Path(BASE_DIR) / REPO_DIR_NAME

def _ensure_repo_on_path() -> Path:
    repo_colab = REPO_DIR / "colab"
    if not REPO_DIR.exists():
        import subprocess
        print(f"Repo no encontrado en {REPO_DIR}. Clonando...")
        clone_cmd = ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)]
        result = subprocess.run(clone_cmd, text=True, capture_output=True, check=False)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f"No se pudo clonar el repo para habilitar import app (code={result.returncode}).")

    repo_colab_str = str(repo_colab)
    if repo_colab_str not in sys.path:
        sys.path.insert(0, repo_colab_str)

    from bootstrap import clone_or_update_repo, configure_sys_path

    repo_root = clone_or_update_repo(REPO_URL, REPO_DIR, BRANCH)
    configure_sys_path(repo_root)
    return Path(repo_root)

try:
    from app.adapters.geostatspy_adapter import GeostatSpyAdapter
    from app.services.geostat_service import GeostatService
except ModuleNotFoundError:
    print("No se pudo importar 'app'. Ejecutando autorecuperación de ruta del repo...")
    _ensure_repo_on_path()
    from app.adapters.geostatspy_adapter import GeostatSpyAdapter
    from app.services.geostat_service import GeostatService

plt.style.use("seaborn-v0_8-whitegrid")

try:
    service  # type: ignore[name-defined]
    print("Se reutiliza `service` existente del notebook anterior.")
except NameError:
    service = GeostatService(adapter=GeostatSpyAdapter())
    print("`service` no existía. Se creó una nueva instancia.")

UPLOAD_DIR = Path("/content/geostat_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("UPLOAD_DIR:", UPLOAD_DIR)
print("REPO_DIR en uso:", REPO_DIR)


In [ ]:
# 1) Parámetros editables (columnas + variografía)
# Si dejas vacío, se usa autodetección.
X_COLUMN_OVERRIDE = ""
Y_COLUMN_OVERRIDE = ""
Z_COLUMN_OVERRIDE = ""   # Si queda vacío, se creará z sintética = 0.0
TARGET_COLUMN_OVERRIDE = ""
DOMAIN_COLUMN_OVERRIDE = ""  # opcional
HOLE_ID_COLUMN_OVERRIDE = ""  # opcional

# Parámetros básicos y prudentes para variografía:
VARIO_LAG_DISTANCE = 10.0
VARIO_N_LAGS = 12
VARIO_LAG_TOLERANCE = 5.0
VARIO_MAX_DISTANCE = 120.0
VARIO_AZIMUTH = 0.0
VARIO_DIP = 0.0
VARIO_ANG_TOL_H = 90.0
VARIO_ANG_TOL_V = 90.0
VARIO_BAND_WIDTH = 0.0
VARIO_BAND_HEIGHT = 0.0
VARIO_ESTIMATOR = "classical"


In [ ]:
# 2) Subir CSV desde memoria local del usuario (Colab uploader)
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se subió ningún archivo. Ejecuta esta celda y selecciona un CSV.")

uploaded_name = list(uploaded.keys())[0]
uploaded_bytes = uploaded[uploaded_name]
csv_path = UPLOAD_DIR / uploaded_name
csv_path.write_bytes(uploaded_bytes)

print("CSV subido:", uploaded_name)
print("Ruta estable:", csv_path)


In [ ]:
# 3) Carga real con GeostatService + columnas disponibles + autodetección
load_result = service.load_csv(str(csv_path))
print("load_csv.success:", load_result.success)
print("load_csv.message:", load_result.message)
print("load_csv.details:", load_result.details)

if not load_result.success:
    raise RuntimeError("Falló load_csv. Revisa el archivo y vuelve a intentar.")

columns = service.get_available_columns()
autodetected = service.get_autodetected_columns()
print("\nColumnas disponibles:")
print(columns)
print("\nAutodetección del servicio:")
print(json.dumps(autodetected, ensure_ascii=False, indent=2))


In [ ]:
# 4) Resolver X/Y/Z/target (autodetección + overrides) y aplicar configuración
# Preferimos lógica real del servicio y sólo permitimos override manual en notebook.
x_col = (X_COLUMN_OVERRIDE or autodetected.get("x") or "").strip()
y_col = (Y_COLUMN_OVERRIDE or autodetected.get("y") or "").strip()
z_col = (Z_COLUMN_OVERRIDE or autodetected.get("z") or "").strip()
target_col = (TARGET_COLUMN_OVERRIDE or autodetected.get("target") or "").strip()
domain_col = (DOMAIN_COLUMN_OVERRIDE or autodetected.get("domain") or "").strip()
hole_id_col = (HOLE_ID_COLUMN_OVERRIDE or autodetected.get("hole_id") or "").strip()

print("Selección inicial:")
print({"x": x_col, "y": y_col, "z": z_col, "target": target_col, "domain": domain_col, "hole_id": hole_id_col})

if not x_col or not y_col or not target_col:
    raise RuntimeError("No se pudo resolver X/Y/target. Define overrides en la celda de parámetros.")

# Manejo seguro cuando Z no aplica:
# GeostatService requiere X/Y/Z/target; si Z está vacío, generamos columna sintética z=0.0.
if not z_col:
    synthetic_z_col = "__z_colab__"
    if service.current_dataset is None:
        raise RuntimeError("No hay dataset cargado para crear Z sintética.")
    df_tmp = service.current_dataset.dataframe.copy()
    df_tmp[synthetic_z_col] = 0.0
    synthetic_csv_path = UPLOAD_DIR / f"{Path(csv_path).stem}__with_z.csv"
    df_tmp.to_csv(synthetic_csv_path, index=False)
    print(f"Z vacía: se creó columna sintética `{synthetic_z_col}` y CSV temporal {synthetic_csv_path}")

    reload_result = service.load_csv(str(synthetic_csv_path))
    print("reload.success:", reload_result.success)
    if not reload_result.success:
        raise RuntimeError(f"No se pudo recargar dataset con Z sintética: {reload_result.message}")

    x_col = X_COLUMN_OVERRIDE or x_col
    y_col = Y_COLUMN_OVERRIDE or y_col
    z_col = synthetic_z_col
    target_col = TARGET_COLUMN_OVERRIDE or target_col

cfg_result = service.set_variable_config(
    x_column=x_col,
    y_column=y_col,
    z_column=z_col,
    target_column=target_col,
    hole_id_column=hole_id_col or None,
    domain_column=domain_col or None,
)
print("\nset_variable_config.success:", cfg_result.success)
print("set_variable_config.message:", cfg_result.message)
print("set_variable_config.eda_summary:", cfg_result.eda_summary)

if not cfg_result.success:
    raise RuntimeError("No se pudo aplicar la configuración X/Y/Z/target.")


In [ ]:
# 5) EDA real (tabla + resumen + gráficos inline)
summary_text = service.build_eda_summary()
stats_rows = service.get_target_statistics_table()
print("Resumen EDA:", summary_text)

if stats_rows:
    display(pd.DataFrame(stats_rows, columns=["metric", "value"]))
else:
    print("No hay tabla de estadísticas disponible.")

try:
    univariate = service.prepare_univariate_data()
    target_values = univariate.get("target_values", [])
    prob_x = univariate.get("probplot_x", [])
    prob_y = univariate.get("probplot_y", [])

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    if target_values:
        axes[0].hist(target_values, bins=30, color="#1f77b4", alpha=0.85)
        axes[0].set_title("Histograma target")
        axes[1].boxplot(target_values, vert=True)
        axes[1].set_title("Boxplot target")
    else:
        axes[0].text(0.5, 0.5, "Sin datos target", ha="center", va="center")
        axes[1].text(0.5, 0.5, "Sin datos target", ha="center", va="center")

    if prob_x and prob_y:
        axes[2].scatter(prob_x, prob_y, s=10, alpha=0.8)
        axes[2].set_title("Probability plot")
        axes[2].set_xlabel("Cuantiles teóricos")
        axes[2].set_ylabel("Valores ordenados")
    else:
        axes[2].text(0.5, 0.5, "Probability no disponible", ha="center", va="center")

    plt.tight_layout()
    plt.show()
except Exception as exc:  # noqa: BLE001
    print(f"EDA no ejecutable con este dataset/configuración: {exc}")


In [ ]:
# 6) Variografía real básica (cálculo + salida técnica + gráfico inline)
variography_params = {
    "target_col": target_col,
    "lag_distance": float(VARIO_LAG_DISTANCE),
    "n_lags": int(VARIO_N_LAGS),
    "lag_tolerance": float(VARIO_LAG_TOLERANCE),
    "max_distance": float(VARIO_MAX_DISTANCE),
    "azimuth": float(VARIO_AZIMUTH),
    "dip": float(VARIO_DIP),
    "ang_tol_h": float(VARIO_ANG_TOL_H),
    "ang_tol_v": float(VARIO_ANG_TOL_V),
    "band_width": float(VARIO_BAND_WIDTH),
    "band_height": float(VARIO_BAND_HEIGHT),
    "estimator": str(VARIO_ESTIMATOR),
}

response = service.compute_experimental_variography(variography_params)
print("ok:", response.ok)
print("message:", response.message)
print("warnings:", [w.code for w in response.warnings])
print("blockers:", [b.code for b in response.blockers])

if response.result is None:
    print("No hay resultado numérico de variografía para graficar.")
else:
    lags = response.result.lag_centers
    gamma = response.result.gamma_values
    pairs = response.result.pair_counts

    display(pd.DataFrame({"lag_center": lags, "gamma": gamma, "npairs": pairs}))

    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(lags, gamma, marker="o", color="#2ca02c", label="gamma(h)")
    ax1.set_xlabel("Lag center")
    ax1.set_ylabel("Semivarianza")
    ax1.set_title("Variograma experimental")

    ax2 = ax1.twinx()
    ax2.bar(lags, pairs, alpha=0.18, width=max(1.0, (max(lags) / max(1, len(lags))) * 0.6), color="#ff7f0e", label="npairs")
    ax2.set_ylabel("N pares")

    plt.tight_layout()
    plt.show()


## Estado final esperado
Si llegaste hasta aquí:
- CSV local cargado desde uploader de Colab.
- Configuración X/Y/Z/target aplicada (con Z sintética si fue necesario).
- EDA ejecutado en modo inline.
- Variografía ejecutada y visualizada en notebook.
